# Layout

> Parent-controlled layout attributes (`[stack]`, `[cluster]`, `[grid]`, `[toolbar]`, `[flow]`, `[grow]`), spacing and flex utilities, and `[responsive-layout]` container queries.

Controls have internal padding and no outer margins. Set spacing between children on the parent container with CSS `gap`. The same control then works in a form, a toolbar, or a panel with no trailing margin on the last item.

Apply layout attributes directly to semantic HTML elements such as `section[stack]`, `nav[cluster]`, or `ul[grid]`. Svelte components and Python constructors can emit these attributes directly without wrapper elements or extra CSS rules.

In [ ]:
from answerui.preview import preview

## Stack, cluster, grid, and toolbar

All rules in `@layer layout` are scoped to `:root` and `[ui]` up to `[unstyle]`:

- `[stack]` arranges children in a vertical flex column stretched to the container width.
- `[cluster]` arranges children in a wrapping horizontal flex row (`flex-wrap: wrap`, `align-items: center`, `justify-content: start`) at their natural widths. Add utility classes such as `.flex-nowrap`, `.items-start`, or `.justify-between` to adjust wrapping or alignment.
- `[grid]` fits as many equal-width columns as possible at `--grid-min` (default `16rem`). In a narrower container, a single column shrinks below `--grid-min`. Direct children have `min-inline-size: 0`.
- `[toolbar]` lays out a single-row, vertically centred flex bar with `min-inline-size: 0` and a `var(--space-2)` gap.

`--gap` is reset to `var(--space)` on every `[stack]`, `[cluster]`, and `[grid]` element. Overriding `--gap` on an outer layout therefore does not change `--gap` inside nested layouts. Applying any of these three attributes to a `ul` or `ol` also clears list padding and markers.

Add `[grow]` to a direct child of `[stack]` or `[cluster]` to fill the remaining height or width (`flex: 1`). Because `[grow]` also sets `min-inline-size: 0` and `min-block-size: 0`, long text in a row wraps instead of crowding sibling controls.

In [ ]:
#| hide
#| asset styles.css
_css = r'''
@layer layout {
  @scope (:root, [ui]) to ([unstyle]) {
    :where(:scope, *):is([stack], [cluster], [grid]) {
      --gap: var(--space); gap: var(--gap);
      &:is(ul, ol) { padding: 0; list-style: none; }
    }
    :where(:scope, *)[stack] { display: flex; flex-direction: column; }
    :where(:scope, *)[cluster] { display: flex; flex-wrap: wrap; align-items: center; justify-content: start; }
    :where(:scope, *)[grid] {
      --grid-min: 16rem; display: grid;
      grid-template-columns: repeat(auto-fit, minmax(min(100%, var(--grid-min)), 1fr));
      > * { min-inline-size: 0; }
    }
    :where(:scope, *):is([stack], [cluster]) > [grow] { flex: 1; min-inline-size: 0; min-block-size: 0; }
    :where(:scope, *)[toolbar] { display: flex; align-items: center; gap: var(--space-2); min-inline-size: 0; }
  }
}
'''

In [ ]:
member = '''<li cluster>
  <div stack grow class="gap-1"><strong>Ada Lovelace</strong><small>Workspace owner</small></div>
  <button type="button" btn btn-variant="secondary">Edit</button>
</li>'''
preview(f'''<section card style="max-inline-size:26rem">
  <ul stack>{member}{member.replace('Ada Lovelace','Grace Hopper').replace('Workspace owner','Editor')}</ul>
</section>''', height='12rem')

```html
<section card style="max-inline-size:26rem">
  <ul stack><li cluster>
  <div stack grow style="--gap:var(--space-3)"><strong>Ada Lovelace</strong><small>Workspace owner</small></div>
  <button type="button" btn btn-variant="secondary">Edit</button>
</li><li cluster>
  <div stack grow style="--gap:var(--space-3)"><strong>Grace Hopper</strong><small>Editor</small></div>
  <button type="button" btn btn-variant="secondary">Edit</button>
</li></ul>
</section>
```

## Flow layout (`[flow]`)

Use `[flow]` for short sections of headings and paragraphs when you want vertical spacing without full `[prose]` typography for lists, code blocks, and tables. `[flow]` sets `display: grid`, `align-content: start`, and `row-gap: var(--flow-space)` (default `.5em`). Any direct child heading (`h1` through `h6`) after the first child has an extra `margin-block-start: .5em`.

`--flow-space` is reset to `.5em` on every `[flow]` element. You can override `--flow-space` on a nested `[flow]` section to tighten its spacing without changing the outer container's row gap.

In [ ]:
#| hide
#| asset styles.css
_css = r'''
@layer layout {
  @scope (:root, [ui]) to ([unstyle]) {
    :where(:scope, *)[flow] {
      --flow-space: .5em; display: grid; align-content: start; row-gap: var(--flow-space);
      > :is(h1, h2, h3, h4, h5, h6):not(:first-child) { margin-block-start: .5em; }
    }
  }
}
'''

In [ ]:
preview('''
<article card flow>
  <h2>Sharing a workspace</h2>
  <p>Invite your colleagues and decide what they can edit.</p>
  <section flow style="--flow-space:.2em">
    <h3>Permissions</h3>
    <p>Owners can invite others.</p>
    <p>Editors can change documents.</p>
  </section>
  <p>Changes apply immediately.</p>
</article>
''', height='16rem')

```html

<article card flow>
  <h2>Sharing a workspace</h2>
  <p>Invite your colleagues and decide what they can edit.</p>
  <section flow style="--flow-space:.2em">
    <h3>Permissions</h3>
    <p>Owners can invite others.</p>
    <p>Editors can change documents.</p>
  </section>
  <p>Changes apply immediately.</p>
</article>

```

## Layout and visibility utilities

Prefer `[stack]`, `[cluster]`, `[grid]`, `[toolbar]`, or `[flow]` for standard layouts. Because `@layer utilities` comes last in the cascade, utility classes such as `.gap-2` or `.flex-nowrap` also override layout defaults on `[stack]`, `[cluster]`, and `[grid]` elements without inline styles. For overrides and one-off arrangements such as `class="flex items-center gap-2"`, `@layer utilities` defines spacing, flex, text, visibility, and responsive helpers:

- Spacing: `.p-*` (`padding`), `.px-*` (`padding-inline`), and `.py-*` (`padding-block`) for steps `0`, `1` through `4`, and `6`, plus `.gap-*` (`gap`) for steps `0`, `1` through `4`, `6`, `8`, and `12`. Step `0` is `0`; steps `1`, `2`, `3`, `4`, `6`, `8`, and `12` map to `--space-3`, `--space-2`, `--space-1`, `--space`, `--space1`, `--space2`, and `--space3`. `.mx-auto` centres a block horizontally (`margin-inline: auto`).
- Flex: `.flex`, `.flex-col`, `.flex-wrap`, `.flex-nowrap`, `.items-start`, `.items-center`, `.justify-center`, `.justify-between`, `.justify-end`, `.flex-1`, `.flex-none`, and `.min-w-0` (`min-inline-size: 0`).
- Text: `.text-center` (`text-align: center`), `.text-muted` (`color: var(--color-muted)`), `.truncate` (single-line ellipsis with `min-inline-size: 0`), and `.tabular-nums` (`font-variant-numeric: tabular-nums`).
- Visibility: `.ui-visually-hidden` hides an element visually while keeping it accessible to screen readers. `.ui-hidden` and `[hidden]:not([hidden=until-found])` set `display: none !important`.

Add `[responsive-layout]` to a container to declare the `ui-layout` container (`container: ui-layout / inline-size`). Below `48rem`, `.ui-wide-only` is hidden with `display: none !important`. At `48rem` and above, `.ui-compact-only` is hidden instead. Neither class sets `display` when the element is visible.

In [ ]:
#| hide
#| asset styles.css
_css = r'''
@layer utilities {
  @scope (:root, [ui]) to ([unstyle]) {
    .ui-visually-hidden { position: absolute; inline-size: 1px; block-size: 1px; padding: 0; overflow: hidden; clip-path: inset(50%); white-space: nowrap; }
    .ui-hidden, :where(:scope, *)[hidden]:not([hidden=until-found]) { display: none !important; }

    .p-0 { padding: 0; }
    .p-1 { padding: var(--space-3); }
    .p-2 { padding: var(--space-2); }
    .p-3 { padding: var(--space-1); }
    .p-4 { padding: var(--space); }
    .p-6 { padding: var(--space1); }
    .px-0 { padding-inline: 0; }
    .px-1 { padding-inline: var(--space-3); }
    .px-2 { padding-inline: var(--space-2); }
    .px-3 { padding-inline: var(--space-1); }
    .px-4 { padding-inline: var(--space); }
    .px-6 { padding-inline: var(--space1); }
    .py-0 { padding-block: 0; }
    .py-1 { padding-block: var(--space-3); }
    .py-2 { padding-block: var(--space-2); }
    .py-3 { padding-block: var(--space-1); }
    .py-4 { padding-block: var(--space); }
    .py-6 { padding-block: var(--space1); }
    .gap-0 { gap: 0; }
    .gap-1 { gap: var(--space-3); }
    .gap-2 { gap: var(--space-2); }
    .gap-3 { gap: var(--space-1); }
    .gap-4 { gap: var(--space); }
    .gap-6 { gap: var(--space1); }
    .gap-8 { gap: var(--space2); }
    .gap-12 { gap: var(--space3); }
    .mx-auto { margin-inline: auto; }
    .flex { display: flex; }
    .flex-col { flex-direction: column; }
    .flex-wrap { flex-wrap: wrap; }
    .flex-nowrap { flex-wrap: nowrap; }
    .items-start { align-items: start; }
    .items-center { align-items: center; }
    .justify-center { justify-content: center; }
    .justify-between { justify-content: space-between; }
    .justify-end { justify-content: end; }
    .flex-1 { flex: 1; }
    .flex-none { flex: none; }
    .min-w-0 { min-inline-size: 0; }
    .text-center { text-align: center; }
    .text-muted { color: var(--color-muted); }
    .truncate { min-inline-size: 0; overflow: hidden; text-overflow: ellipsis; white-space: nowrap; }
    .tabular-nums { font-variant-numeric: tabular-nums; }
    [responsive-layout] { container: ui-layout / inline-size; }
    @container ui-layout (width < 48rem) { .ui-wide-only { display: none !important; } }
    @container ui-layout (width >= 48rem) { .ui-compact-only { display: none !important; } }
  }
}
'''